# Unpacking Waveforms: Saving to MiniSEED

Welcome back! In our last notebook, we discovered that our raw `.m` files are actually like little suitcases packed with multiple recordings (Traces) all bundled together.

Today, our goal is to **unpack the suitcase**. We are going to open every single `.m` file, pull out the individual Traces one by one, and save each one into its own separate file.

We will save them using the **MiniSEED** format (`.mseed`). MiniSEED is the gold standard format in seismology—it's highly compressed and makes it very easy for machine learning programs to read specific stations without having to open the whole suitcase!

### Step 1: Setting up our workspace
First, we import our tools and tell Python where our data lives. 
We also tell Python to create a brand new folder to store all our unpacked files safely.

In [1]:
import obspy
import glob
import os
import numpy as np

# Where are the suitcases?
waveform_dir = 'top_300_raw_cut_waveforms/'

# Where do we put the unpacked clothes?
output_dir = 'unpack_top_300_miniseed_raw/'

# Ask the computer to create the folder if it doesn't exist yet
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created a brand new folder: {output_dir}")
else:
    print(f"Folder {output_dir} is ready to go!")

Folder unpack_top_300_miniseed_raw/ is ready to go!


### Step 2: The Unpacking Loop
This is the main event! Here is what we are doing in the cell below:
1. **Find** all the `.m` files.
2. **Open** each file using `obspy` (this gives us the bundle of Traces).
3. **Loop** through every single Trace inside the bundle.
4. **Name** the new file carefully so we don't accidentally overwrite data. We use the Network, Station, Channel, and exact Timestamps.
5. **Save** the trace in the new `.mseed` format.

In [2]:
# To keep this notebook fast, we will only unpack the VERY FIRST file in our list!
m_files = glob.glob(os.path.join(waveform_dir, '*.m'))

if m_files:
    # Grab the first file
    file_path = m_files[0]
    print(f"Unpacking file: {os.path.basename(file_path)}")
    
    # Read the bundle (Stream)
    st = obspy.read(file_path)
    print(f"Found {len(st)} traces inside. Extracting now...")
    
    # Extract each trace!
    for tr in st:
        # Grab metadata for the filename
        # If a network code is missing, we use 'XX' as a placeholder
        network = tr.stats.network if tr.stats.network else "XX"
        station = tr.stats.station
        channel = tr.stats.channel
        
        # Format times neatly so they work safely as filenames (no weird colons!)
        start_time_str = tr.stats.starttime.strftime('%Y%m%dT%H%M%S')
        end_time_str = tr.stats.endtime.strftime('%Y%m%dT%H%M%S')
        
        # Create the perfect, descriptive filename
        filename = f"{network}.{station}.{channel}.{start_time_str}.{end_time_str}.mseed"
        output_path = os.path.join(output_dir, filename)
        
        # Save the file! (This is the magic step)
        tr.write(output_path, format="MSEED")
        
    print("Finished unpacking!")

Unpacking file: 20200713063040900.m
Found 21 traces inside. Extracting now...
Finished unpacking!


### Step 3: Proving We Didn't Break Anything (Verification)
In science and programming, you should always double-check your work!
Did converting the file from `.m` to `.mseed` accidentally delete our important metadata? Did it corrupt the physical wave recordings?

Let's write a script to compare the **original trace** to the **newly unpacked trace**.

In [3]:
if m_files:
    # Let's look at the very first trace from our original bundle
    orig = obspy.read(m_files[0])
    orig_tr = orig[0]
    
    # We know exactly what its new filename should be!
    network = orig_tr.stats.network if orig_tr.stats.network else "XX"
    station = orig_tr.stats.station
    channel = orig_tr.stats.channel
    start_time_str = orig_tr.stats.starttime.strftime('%Y%m%dT%H%M%S')
    end_time_str = orig_tr.stats.endtime.strftime('%Y%m%dT%H%M%S')
    expected_filename = f"{network}.{station}.{channel}.{start_time_str}.{end_time_str}.mseed"
    
    # Read the unpacked version
    unpacked_path = os.path.join(output_dir, expected_filename)
    unp = obspy.read(unpacked_path)
    unp_tr = unp[0]
    
    print("--- VERIFICATION TEST ---")
    # 1. Did the metadata survive?
    keys_to_check = ['network', 'station', 'channel', 'starttime', 'endtime', 'sampling_rate', 'npts']
    metadata_match = all(orig_tr.stats[k] == unp_tr.stats[k] for k in keys_to_check)
    print(f"Does the Metadata match? {'✅ Yes' if metadata_match else '❌ No'}")
    
    # 2. Did the physical wiggles (data points) survive?
    # We use numpy (np) to compare the giant lists of numbers quickly
    data_match = np.array_equal(orig_tr.data, unp_tr.data)
    print(f"Does the Physical Waveform Data match? {'✅ Yes' if data_match else '❌ No'}")

--- VERIFICATION TEST ---
Does the Metadata match? ✅ Yes
Does the Physical Waveform Data match? ✅ Yes


---
**Congratulations!** 
You now know how to unpack complex bundles of earthquake data, safely export them into industry-standard formats, and programmatically verify that your data translation was flawless!